In [0]:
%run /Repos/dung_nguyen_hoang@mfcgd.com/Utilities/Functions

In [0]:
import pyspark.sql.functions as F

In [0]:
df = spark.read.format("parquet").load(
    "/mnt/lab/vn/project/VN_NB/NB_UW/"
).filter(
    (F.col("CHANNEL") == "AGENCY")
    &(F.col("SUBMIT_TYPE") != "SIO")
)

df = df.withColumn(
    "EXISTING_IND",
    F.greatest(F.col("EXISTING_PO_IND"), F.col("EXISTING_LA_IND"))
).withColumn(
    "ILP_IND",
    F.when(F.col("BENEFIT_BASIC").like("RUV%"), 1)
    .otherwise(0)
).withColumn(
    "UL_IND",
    F.when(F.col("BENEFIT_BASIC").like("UL%"), 1)
    .otherwise(0)
).withColumn(
    "TROP_IND",
    F.when(F.col("BENEFIT_BASIC").like("RP%"), 1)
    .otherwise(0)
).withColumn(
    "OTHER_IND",
    F.when(
        ~F.col("BENEFIT_BASIC").like("RUV%") & 
        ~F.col("BENEFIT_BASIC").like("UL%") & 
        ~F.col("BENEFIT_BASIC").like("RP%"), 
        1
    ).otherwise(0)
).withColumn(
    "MC_RIDER_IND",
    F.when(F.col("BENEFIT_RIDER").like("%MC%"), 1)
    .otherwise(0)
).withColumn(
    "HC_RIDER_IND",
    F.when(F.col("BENEFIT_RIDER").like("%HC%"), 1)
    .otherwise(0)
).withColumn(
    "CI_RIDER_IND",
    F.when(F.col("BENEFIT_RIDER").like("%CI%"), 1)
    .otherwise(0)
).withColumn(
    "OTHER_RIDER_IND",
    F.when(
        ~F.col("BENEFIT_RIDER").like("%MC%") & 
        ~F.col("BENEFIT_RIDER").like("%HC%") & 
        ~F.col("BENEFIT_RIDER").like("%CI%"), 
        1
    ).otherwise(0)
).withColumn(
    "NO_RIDER_IND",
    F.when(
        F.col("BENEFIT_RIDER").isNull() | 
        (F.length(F.col("BENEFIT_RIDER")) == 0), 
        1
    ).otherwise(0)
).withColumn(
    "AGENT_TIER",
    F.when(F.col("AGENT_TIER") == "MDRT", "1.MDRT")
    .when(F.col("AGENT_TIER") == "P", "2.Platinum")
    .when(F.col("AGENT_TIER") == "G", "3.Gold")
    .when(F.col("AGENT_TIER") == "S", "4.Silver")
    .otherwise("5.Other/Non-ManuPro")
).withColumn(
    "PO_TENURE_cat",
    F.when(F.col("PO_TENURE") < 1, "1.< 1yr")
    .when(F.col("PO_TENURE") < 2, "2.1 - 2yr")
    .when(F.col("PO_TENURE") < 3, "3.2 - 3yr")
    .when(F.col("PO_TENURE") < 5, "4.3 - 5yr")
    .otherwise("5.> 5yr")
).withColumn(
    "LA_TENURE_cat",
    F.when(F.col("LA_TENURE") < 1, "1.< 1yr")
    .when(F.col("LA_TENURE") < 2, "2.1 - 2yr")
    .when(F.col("LA_TENURE") < 3, "3.2 - 3yr")
    .when(F.col("LA_TENURE") < 5, "4.3 - 5yr")
    .otherwise("5.> 5yr")
).withColumn(
    "LOADING",
    F.when(F.col("LOADING").isNotNull() | (F.length(F.col("LOADING")) > 0), "Y")
    .otherwise("N")
)

df_nb = df.select(
    "POL_NUM", "PO_NUM", "PO_TENURE", "LA_TENURE", "AGENT_TIER", "PO_TENURE_cat", "LA_TENURE_cat",
    "BENEFIT_BASIC", "BENEFIT_RIDER", "POLICY_STATUS", "EXISTING_IND", "EXISTING_LA_IND", 
    "LOADING", "TOT_APE", "ILP_IND", "UL_IND", "TROP_IND", "OTHER_IND",
    "MC_RIDER_IND", "HC_RIDER_IND", "CI_RIDER_IND", "OTHER_RIDER_IND", "NO_RIDER_IND", 
    "STP_FLAG", "AIS_DECISION", "ASSIGN_TEAM", "ADMIN_RULE_CATEGORY", "ADMIN_REMARKS",
    "AIS_RULE_CATEGORY", "DIGITEXX_REMARK_CATEGORY",
    "CHANNEL", "image_date"
).dropDuplicates()

# df.groupBy(
#     "POLICY_STATUS"
# ).agg(
#     F.countDistinct(F.col("POL_NUM")).alias("policy_count"),
#     #F.sum(F.when(F.col("POLICY_STATUS") == F.lit("REJECTED"), 1).otherwise(0)).alias("reject_count"),
#     #F.sum(F.when(F.col("POLICY_STATUS") == F.lit("DECLINED"), 1).otherwise(0)).alias("decline_count"),
#     #F.sum(F.when(F.col("POLICY_STATUS") == F.lit("NOT-TAKEN"), 1).otherwise(0)).alias("ntk_count"),
#     #F.sum(F.when(F.col("POLICY_STATUS") == F.lit("APPROVED"), 1).otherwise(0)).alias("reject_count")
# ).display()

# df_ape.groupBy(
#     "image_date",
#     "CHANNEL",
#     "POLICY_STATUS",
#     "EXISTING_IND",
#     "EXISTING_LA_IND"
# ).agg(
#     F.countDistinct(F.col("PO_NUM")).alias("po_count"),
#     F.countDistinct(F.col("POL_NUM")).alias("policy_count"),
#     F.sum(F.col("TOT_APE")).alias("ape")
# ).orderBy(
#     "image_date",
#     "CHANNEL",
#     F.desc("EXISTING_IND"),
#     F.desc("policy_count")
# ).display()
df_final = df.select(*[col.lower() for col in df.columns])
#df_final.toPandas().to_csv("/dbfs/mnt/lab/vn/project/VN_NB/CSV_FILES/VN_NB_analysis_rawdata_v3.csv", index=False, header=True, encoding='utf-8-sig')
df_nb_filtered = df_nb.filter(F.col("POLICY_STATUS") == "APPROVED")

In [0]:
df_final.filter(F.col("pol_num")=="3828630507").display()

In [0]:
df_nb_filtered.groupBy(
    "image_date",
    "existing_ind",
    "existing_la_ind"
).agg(
    F.count(F.col("pol_num")).alias("policy_count"),
    F.sum(F.col("tot_ape")).alias("ape")
).orderBy(
    "image_date",
    F.desc("existing_ind"),
    F.desc("existing_la_ind")
).display()

In [0]:
df_final.groupBy(
    "image_date",
    "policy_status"
).agg(
    F.count(F.col("pol_num")).alias("policy_count")
).orderBy(
    "image_date"
).display()

In [0]:
df_filtered_pd = df_nb_filtered.toPandas()
df_filtered_pd.head(2)

In [0]:
import pandas as pd

def create_policy_table(df, existing_ind_value, rows_arr, cols_arr):
    # Filter for the specified EXISTING_IND
    df_customers = df[df['EXISTING_IND'] == existing_ind_value].copy()

    # Check if filtered DataFrame is empty
    if df_customers.empty:
        print(f"No data found for EXISTING_IND == '{existing_ind_value}'")
        return pd.DataFrame(columns=['Policy Type'] + cols_arr)  # Return an empty DataFrame if no data is found

    # Create a list to collect results for each policy type
    results = []

    # Iterate over each policy type column and calculate counts
    for policy_column in rows_arr:
        # Determine the policy name from the column name
        policy_name = policy_column.replace('_IND', '')

        # Filter for rows where the indicator is 1
        df_filtered = df_customers[df_customers[policy_column] == 1]

        # Check filtered DataFrame shape
        #print(f"Filtered DataFrame shape for {policy_name}:", df_filtered.shape)

        # Group by image_date and count unique POL_NUM
        df_grouped = df_filtered.groupby('image_date')['POL_NUM'].nunique().reset_index()

        # Add the Policy Type to the results
        df_grouped['Policy Type'] = policy_name
        results.append(df_grouped)

    # Combine results into a single DataFrame
    df_combined = pd.concat(results, ignore_index=True)

    # Pivot the DataFrame to get the desired format
    table = df_combined.pivot(index='Policy Type', columns='image_date', values='POL_NUM').fillna(0)

    # Reindex to ensure all specified dates are present
    table = table.reindex(columns=cols_arr, fill_value=0)

    # Reset index to have a clean DataFrame
    table = table.reset_index()

    return table

# Extract unique values from the image_date column
cols_arr = sorted(df_filtered_pd['image_date'].unique())

In [0]:
# Define the rows array based on the columns to be shown in rows
rows_arr = ['ILP_IND', 'UL_IND', 'TROP_IND', 'OTHER_IND']

# Create table for new customers (EXISTING_IND == 'N')
table_new_customers = create_policy_table(df_filtered_pd, 'N', rows_arr, cols_arr)

# Display the tables
print("New Customers Policy Table:")
display(table_new_customers)

In [0]:
# Define the rows array based on the columns to be shown in rows
rows_arr = ['ILP_IND', 'UL_IND', 'TROP_IND', 'OTHER_IND']
# Create table for existing customers (EXISTING_IND == 'Y')
table_existing_customers = create_policy_table(df_filtered_pd, 'Y', rows_arr, cols_arr)

print("\nExisting Customers Policy Table:")
display(table_existing_customers)

In [0]:
# Define the rows array based on the columns to be shown in rows
rows_arr = ['MC_RIDER_IND', 'HC_RIDER_IND', 'CI_RIDER_IND', 'OTHER_RIDER_IND', 'NO_RIDER_IND']

# Create table for new customers (EXISTING_IND == 'N')
table_new_customers = create_policy_table(df_filtered_pd, 'N', rows_arr, cols_arr)

# Display the tables
print("New Customers Rider Table:")
display(table_new_customers)

In [0]:
# Define the rows array based on the columns to be shown in rows
rows_arr = ['MC_RIDER_IND', 'HC_RIDER_IND', 'CI_RIDER_IND', 'OTHER_RIDER_IND', 'NO_RIDER_IND']

# Create table for existing customers (EXISTING_IND == 'Y')
table_new_customers = create_policy_table(df_filtered_pd, 'Y', rows_arr, cols_arr)

# Display the tables
print("Existing Customers Rider Table:")
display(table_new_customers)

In [0]:
# Define the rows array based on the columns to be shown in rows
rows_arr = ['MC_RIDER_IND', 'HC_RIDER_IND', 'CI_RIDER_IND', 'OTHER_RIDER_IND', 'NO_RIDER_IND']

# Create table for existing customers (EXISTING_IND == 'Y')
table_new_customers = create_policy_table(df_filtered_pd, 'Y', rows_arr, cols_arr)

# Display the tables
print("Existing Customers Rider Table:")
display(table_new_customers)

In [0]:
df_nb_filtered.groupBy(
    "image_date",
    "EXISTING_IND",
    "EXISTING_LA_IND",
    "AGENT_TIER",
    "PO_TENURE_cat",
    "LA_TENURE_cat"
).agg(
    F.count("POL_NUM").alias("policy_count")
).orderBy(
    F.desc("EXISTING_IND"),
    F.desc("image_date"),
    "AGENT_TIER"
).display()

In [0]:
df_nb_top_admin_rules = df_nb_filtered.withColumn(
    "UNDERPAYMENT_IND",
    F.when(
        F.col("ADMIN_REMARKS").like("Underpayment%") &
        ~F.col("ADMIN_REMARKS").like("%>%"), "Y")
    .otherwise("N")
)

df_nb_top_admin_rules.withColumn(
    "AIS_DECISION",
    F.when(F.col("STP_FLAG") == True, "Clean")
    .when(F.col("ADMIN_RULE_CATEGORY").isNotNull() |
          F.col("AIS_RULE_CATEGORY").isNotNull() |
          F.col("DIGITEXX_REMARK_CATEGORY").isNotNull(), "Refer to UW")
    .otherwise("Clean")
)

df_nb_top_admin_rules.filter(
    (F.col("CHANNEL") == "AGENCY")
    #&(F.col("STP_FLAG") != True)
    #&(F.col("AIS_DECISION") != "Clean")
    #&(F.length(F.col("ADMIN_RULE_CATEGORY")) > 1)
    #&(F.col("EXISTING_LA_IND") == "Y")
).groupBy(
    #"image_date",
    "EXISTING_LA_IND",
    "STP_FLAG",
    "AIS_DECISION",
    "ADMIN_RULE_CATEGORY",
    "UNDERPAYMENT_IND"
).agg(
    F.count("POL_NUM").alias("policy_count")
).orderBy(
    #F.desc("EXISTING_IND"),
    #"image_date",
    F.desc("policy_count")
).display()


In [0]:
df_rej = df_nb.filter(
    (F.col("POLICY_STATUS").isin(["REJECTED","DECLINED","NOT-TAKEN"]))
    #&(F.col("CHANNEL") == F.lit("AGENCY"))
)

# df_iss = df.filter(
#     (F.col("POLICY_STATUS") == F.lit("APPROVED"))
#     #&(F.col("CHANNEL") == F.lit("AGENCY"))
# )
# print("Rejected: ", df_rej.count(), df_rej.select("POL_NUM").distinct().count())
# print("Approved: ", df_iss.count(), df_iss.select("POL_NUM").distinct().count())

# Deep-dive analysis start here

In [0]:
#df_rej.display()
df_rej.toPandas().to_csv("/dbfs/mnt/lab/vn/project/VN_NB/CSV_FILES/VN_NB_reject_rawdata_v3.csv", index=False, header=True, encoding='utf-8-sig')